# 03 Optional TARDIS Modeling

This notebook is intentionally narrow.  The literature review says sparse spectra should not be over-modeled; TARDIS is useful here as a line-identification and qualitative shape-comparison tool after classification, phase, and velocity have been checked.

Run this in the `tardis` environment only when the target has a usable observed spectrum and a generated TARDIS output under `output/<target>/tardis/`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TARGET = "SN2026jlm"
OUTPUT_DIR = PROJECT_ROOT / "output" / TARGET
SPECTRUM_DIR = OUTPUT_DIR / "spectrum"
TARDIS_DIR = OUTPUT_DIR / "tardis"

## Load observed and simulated spectra

In [ ]:
observed_files = sorted(SPECTRUM_DIR.glob("*.dat"))
simulated_files = sorted(TARDIS_DIR.glob("tardis_spectrum_*.dat"))

if not observed_files:
    raise FileNotFoundError(f"No observed *.dat spectra in {SPECTRUM_DIR}")
if not simulated_files:
    raise FileNotFoundError(f"No TARDIS spectra in {TARDIS_DIR}")

observed_path = observed_files[0]
simulated_path = simulated_files[0]
obs = np.loadtxt(observed_path)
sim = np.loadtxt(simulated_path)

wave_obs, flux_obs = obs[:, 0], obs[:, 1]
wave_sim, flux_sim = sim[:, 0], sim[:, 1]

print(f"Observed:  {observed_path}")
print(f"Simulated: {simulated_path}")

## Normalize and compare shapes

In [ ]:
def normalize(flux):
    finite = np.isfinite(flux)
    scale = np.nanpercentile(np.abs(flux[finite]), 95) if finite.any() else 1.0
    return flux / scale if scale else flux

plt.figure(figsize=(10, 5))
plt.plot(wave_obs, normalize(flux_obs), lw=0.9, label="Observed WISeREP/BFOSC spectrum")
plt.plot(wave_sim, normalize(flux_sim), lw=0.9, label="TARDIS synthetic spectrum")
plt.xlim(3500, 9000)
plt.xlabel("Rest wavelength (Angstrom)")
plt.ylabel("Normalized flux / luminosity density")
plt.title(f"{TARGET}: qualitative TARDIS comparison")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Interpretation boundary

Use this comparison to discuss whether the model broadly matches line locations or continuum shape.  Do not claim strong ejecta mass, explosion energy, or abundance constraints from one sparse spectrum without additional modeling and uncertainties.